In [ ]:
from pathlib import Path

import hardware
import lightning as L
import torch
from env import SKILLS, DuckEnv
from model import ActorCritic
from rollout import evaluate, record_video
from torch.utils.data import DataLoader, IterableDataset

In [ ]:
hw = hardware.load("2026-11-07-hd1910")
env = DuckEnv(SKILLS["walk"], hw, num_envs=2048, device="cuda", seed=1)
obs, critic_obs = env.reset()
policy = ActorCritic(obs.shape[1], critic_obs.shape[1], 14).to("cuda")   # 课程仓库的网络；REINFORCE 只用其中的 actor

In [ ]:
def reward_to_go(rewards, dones, gamma=0.99):
    """从后往前累加折扣回报，回合结束处截断。rewards、dones 的形状是（步数，环境数）。"""
    returns = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[0])
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running * (1 - dones[t])
        returns[t] = running
    return returns

In [ ]:
class Rollout(IterableDataset):
    """用当前策略采一段轨迹，整理成一批：观测、动作、优势。"""

    def __init__(self, env, policy, steps=24):
        super().__init__()
        self.env, self.policy, self.steps = env, policy, steps
        self.batch = None

    def __iter__(self):
        if self.batch is None:                           # Lightning 每轮会读两次数据，只采一次样
            self.batch = self.sample()
        yield self.batch

    def sample(self):
        obs, critic_obs = self.env.get_observations()
        trajectory = []
        for _ in range(self.steps):
            with torch.no_grad():
                actions = self.policy.act(obs, critic_obs)[0]
            next_obs, critic_obs, rewards, dones, _ = self.env.step(actions)
            trajectory.append((obs, actions, rewards, dones.float()))
            self.policy.actor_normalizer.update(next_obs)
            obs = next_obs
        obs, actions, rewards, dones = (torch.stack(x) for x in zip(*trajectory))
        returns = reward_to_go(rewards, dones)
        advantages = (returns - returns.mean()) / (returns.std() + 1e-8)   # 减整批平均当基线，再统一尺度
        return {"obs": obs.flatten(0, 1), "actions": actions.flatten(0, 1),
                "advantages": advantages.flatten(), "reward": rewards.mean()}

In [ ]:
batch = Rollout(env, policy).sample()
{k: tuple(v.shape) for k, v in batch.items()}            # 2048 只鸭子 × 24 步 = 49152 条

In [ ]:
class Reinforce(L.LightningModule):
    """REINFORCE：损失是 −log π(a|s) × 优势，只更新 actor 与动作标准差。"""

    def __init__(self, env, policy, lr=3e-4, entropy_coef=0.01):
        super().__init__()
        self.env, self.policy = env, policy
        self.lr, self.entropy_coef = lr, entropy_coef

    def train_dataloader(self):
        return DataLoader(Rollout(self.env, self.policy), batch_size=None)   # 每轮重建：用最新的策略重新采样

    def training_step(self, batch, batch_idx):
        dist = self.policy.action_distribution(batch["obs"])
        log_prob = dist.log_prob(batch["actions"]).sum(-1)
        entropy = dist.entropy().sum(-1).mean()
        self.log("reward", batch["reward"], prog_bar=True)
        return -(log_prob * batch["advantages"]).mean() - self.entropy_coef * entropy

    def configure_optimizers(self):
        return torch.optim.Adam(self.policy.actor_parameters(), lr=self.lr)

In [ ]:
untrained = evaluate(policy.act_inference, SKILLS["walk"], hw, 0, num_envs=16)   # 训练前先测一次，当对照
untrained["falls"]

In [ ]:
trainer = L.Trainer(max_epochs=1000, reload_dataloaders_every_n_epochs=1, gradient_clip_val=1.0,
                    logger=False, enable_checkpointing=False)
trainer.fit(Reinforce(env, policy))

In [ ]:
policy.to("cuda")                                        # Lightning 训完把模型放回了 CPU
counter = env.unwrapped.common_step_counter              # 课程表走到的那一档，评测对齐到它
trained = evaluate(policy.act_inference, SKILLS["walk"], hw, counter, num_envs=16)
record_video(policy.act_inference, SKILLS["walk"], hw, counter, Path("reinforce.mp4"))
trained["falls"]

In [ ]:
torch.save(policy.state_dict(), "reinforce.pt")          # 下一讲接着用